<a href="https://colab.research.google.com/github/zastrozhnayayana/nn-zero-to-hero-notes/blob/main/02_makemore_bigrams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
words = open('names.txt', 'r').read().splitlines()

--2026-05-27 15:15:53--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt.9’

names.txt.9         100%[===================>] 222.80K  --.-KB/s    in 0.01s   

2026-05-27 15:15:53 (16.2 MB/s) - ‘names.txt.9’ saved [228145/228145]



In [48]:
import torch

stoi = {chr(i): i - ord('a') + 1 for i in range(ord('a'), ord('z') + 1)}
itos = {i - ord('a') + 1: chr(i) for i in range(ord('a'), ord('z') + 1)}
stoi['.'] = 0
itos[0] = '.'

In [49]:
xs, ys = [], []
for w in words:
  w = '.' + w + '.'
  for ch1, ch2 in zip(w, w[1:]):
    xs.append(stoi[ch1])
    ys.append(stoi[ch2])

xs = torch.tensor(xs) # если первый символ такой
ys = torch.tensor(ys) # то следующий такой (это наш датасет)
print(xs, ys)
num_test = len(xs)

tensor([ 0,  5, 13,  ..., 25, 26, 24]) tensor([ 5, 13, 13,  ..., 26, 24,  0])


In [50]:
import torch.nn.functional as F # модуль pytorch с готовыми функциями для нейросетей

xenc = F.one_hot(xs, num_classes=27).float() # создаёт тензор из массивов размера 27, у которых на местах их xs стоят единицы
print("xenc.shape ", xenc.shape, '\n', xenc, sep="") # тензор размера кол-во примеров в датасете на размер алфавита
g = torch.Generator().manual_seed(2147483647)
W = torch.rand((27, 27), generator=g, requires_grad=True) # для каждого из 27 символов вероятности каждого символа быть следующим
# изначально веса произвольные

xenc.shape torch.Size([228146, 27])
tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 1., 0., 0.]])


In [51]:
for k in range(1000):
  #forward pass
  logits = xenc @ W # по сути просто для каждого символа строку, соответствующую его номеру
  logits = logits.exp() # экспоненцируем, чтобы числа стали > 0
  probs = logits / logits.sum(1, keepdim=True) # получаем вероятности
  loss = -probs[torch.arange(num_test), ys].log().mean() + 0.01*(W**2).mean()
  # берём значения вероятностей, перемножаем - это likelihood (правдоподобие, насколько вероятны эти y при таких х)
  # наша цель - максимизировать правдоподобие
  # так как вероятности, которые мы перемножаем < 1, чтобы не уйти в 0 из-за неточности питона в операциях, будем максимизировать логарифм правдоподобия
  # то есть сумму логарифмов вероятностей получить такие y при таких х
  # чтобы мы могли менять кол-во примеров в датасете, а loss оставался в той же размерности, делим на кол-во примеров в датасете
  # (теперь не зависим от размера датасета)
  # и ещё ставим -, теперь мы стремимся минимизировать loss

  #backward pass
  # ставим градиет = None, так как будем делать self.grad += d[self.data]/d[out.data] * out.grad для каждого out, полученного из self
  # показываем, что градиента здесь вообще нет (не то, что он 0)
  W.grad = None
  loss.backward()

  #update
  # изменяем значения всех весов на -lr * W.grad, так как градиент показывает, на сколько delta_x увеличится loss, если увеличить W.data на delta_x, а мы хотим уменьшить loss
  # то есть я всеми параметрами моей нейронной сети делаю маленький шаг в сторону уменьшения loss. утверждается, что такими действиями я в конце концов достигну минимума ф-ии потерь
  # loss(W). W - это переменные. ищу минимум такой ф-ии с помощью градиентного спуска
  W.data += -50 * W.grad
print(loss.item())

2.4805643558502197


In [52]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
  out = []
  ix = 0
  while True:
    xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    p = counts / counts.sum(1, keepdims=True)
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item() # выбирает один индекс из 27 c вероятностями из p
    # .item() убирает тензор
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out))

cexze.
momasurailezityha.
konimittain.
llayn.
ka.
